In [1]:
pip install langchain-google-genai pypdf gradio langchainhub chromadb langchain-core

In [2]:
!pip install langchain-community

In [3]:
import os
import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Set your Gemini API key securely
os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"


rag_chain = None

def initialize_rag_system(pdf_file_path):
    """
    Loads the PDF, chunks it, embeds it into ChromaDB via Google Embeddings,
    and builds the RAG chain using a Gemini Chat Model.
    """
    global rag_chain

    if pdf_file_path is None:
        return "❌ Error: No file uploaded. Please upload a PDF first."

    try:

        path_str = pdf_file_path.name if hasattr(pdf_file_path, 'name') else str(pdf_file_path)


        loader = PyPDFLoader(path_str)
        docs = loader.load()

        if not docs:
            return "❌ Error: The PDF appears to be empty or unscannable images without text layer."

        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        splits = splitter.split_documents(docs)


        embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
        vector_store = Chroma.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever(search_kwargs={"k": 4})


        prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}

Answer:""")


        llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)


        rag_chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )
        return "Database created successfully! Start chatting below."

    except Exception as e:
        return f"❌ System Error: {str(e)}"

def chat_function(message, history):
    """
    Feeds messages directly to the active Gemini RAG architecture
    """
    global rag_chain
    if rag_chain is None:
        return "Please upload and process a PDF file first."

    try:
        response = rag_chain.invoke(message)
        return response
    except Exception as e:
        return f"An error occurred: {str(e)}"


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.HTML("""
        <div style="
            background: linear-gradient(135deg, #4285F4, #34A853);
            padding: 20px;
            border-radius: 10px;
            text-align: center;
            margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        ">
            <h1 style="color: white; margin: 0; font-family: 'Arial', sans-serif; font-size: 28px;">
                PDF Chatbot
            </h1>
            <p style="color: #e0e0e0; margin: 5px 0 0 0; font-size: 14px;">
                Upload any PDF to convert it into a Summary Chatbot.
            </p>
        </div>
    """)



    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Step 1: Upload PDF", file_types=[".pdf"])
            load_btn = gr.Button("Step 2: Train Chatbot", variant="primary")
            status_output = gr.Textbox(label="Training Status", interactive=False)

        with gr.Column(scale=2):
            chat_interface = gr.ChatInterface(
                fn=chat_function,
                type="messages"
            )


    load_btn.click(
        fn=initialize_rag_system,
        inputs=pdf_input,
        outputs=status_output
    )


if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_4483/1455997754.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/tmp/ipykernel_4483/1455997754.py:91: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0a359bb4e72b599e8e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
